### AITW-wood_stiffness_homogenization

Multiscale stiffness of native, polymer-infiltrated, and delignified + infiltrated wood. It works from the cell-wall constituents up to clearwood. 

#### Links

- <a href="https://github.com/AI-TranspWood/Wood_Stiffness_Homogenization" target="_blank" rel="noopener noreferrer">Source code</a>
- <a href="https://www.eessi.io/docs/available_software/detail/AITW-wood_stiffness_homogenization" target="_blank" rel="noopener noreferrer">EESSI</a>


In [ ]:
from utilities import initialize_with_module

await initialize_with_module('AITW-wood_stiffness_homogenization/1.0.0-gfbf-2025b', '2025.06')

In [ ]:
import re
from io import BytesIO

import numpy as np
import pandas as pd
from IPython.display import Image as IPyImage
import ipywidgets as wdg
import matplotlib.pyplot as plt


from wood_stiffness_homogenization import constants as cst
from wood_stiffness_homogenization.cli.postproc import SHORT_TITLE, BARS, COLOR

BARS = ['E_R', 'E_T', 'E_L', 'G_TL', 'G_RL', 'G_RT', 'M_cw']
COLOR = {1: '#8c5a2b', 2: '#1f77b4', 3: '#2ca02c'}  # native / infiltrated / delignified + infiltrated


def output_visualizer(data: pd.DataFrame):
    """
    Visualize the output data from the wood stiffness homogenization process.
    Show the CSV output and a series of plot with a slide bar for each unique property
    
    Parameters:
    - data: A pandas DataFrame containing the output data to visualize.
    """

    unq_map = {}
    for col in cst.INPUTS.keys():
        unq_map[col] = np.unique(data[col])
    state_rev_map = {v: k for k, v in cst.STATES.items()}

    sliders = {}
    for col, unqs in unq_map.items():
        if len(unqs) > 1:
            sliders[col] = wdg.SelectionSlider(
                options=[(f"{unq}", unq) for unq in unqs],
                value=unqs[0],
                description=col,
                continuous_update=False,
            )

    output_box = wdg.Output()

    def display_for_selection(*args, **kwargs):
        # Filter the data based on the current slider values
        output_box.clear_output(wait=True)
        with output_box:
            combo = [(col, val.value) for col, val in sliders.items()]
            filtered_data = data.copy()
            for col, val in combo:
                filtered_data = filtered_data[filtered_data[col] == val]

            if filtered_data.empty:
                return

            _, ax = plt.subplots(figsize=(6, 3.5))

            width = 0.8 / 3
            for state in filtered_data['state']:
                state_idx = state_rev_map[state]
                bar_positions = np.arange(len(BARS)) + (state_idx - 2) * width
                state_row = filtered_data[filtered_data['state'] == state]
                bar_values = [state_row[bar].values[0] for bar in BARS]
                ax.bar(
                    bar_positions, bar_values, width=width,
                    label=f"State: {state}",
                    color=COLOR[state_idx]
                )
            rgx = re.compile(r'([a-zA-Z]+)_([a-zA-Z]+)')
            bar_labels = [rgx.sub(r'\1$_{\2}$', bar) for bar in BARS]
            ax.set_xticks(range(len(BARS)), bar_labels)
            ax.set_ylabel('[GPa]')
            ax.grid(alpha=0.3, axis='y')

            title = f"{', '.join(f'{SHORT_TITLE.get(col, col)}={val}' for col, val in combo)}"
            ax.set_title(title, wrap=True)

            ax.legend()

            image_buffer = BytesIO()
            plt.savefig(image_buffer, format='png', dpi=200)
            image_buffer.seek(0)

            display(IPyImage(data=image_buffer.getvalue(), format='png'))

            # Display the CSV output
            display(filtered_data)

    for col, slider in sliders.items():
        slider.observe(display_for_selection, names='value')
    
    # Group sliders in a horizontal box of vboxes with 3 sliders each
    slider_box = wdg.HBox([wdg.VBox(list(sliders.values())[i:i+3]) for i in range(0, len(sliders), 3)])
    # slider_box = wdg.VBox(list(sliders.values()))

    display_for_selection()  # Display the initial output based on the default slider values

    main_box = wdg.VBox([slider_box, output_box])
    display(main_box)
        

In [ ]:
import ipywidgets as wdg
import utilities
from common import WOOD_SITFFNESS_HOMOG_DEFAULTS, HOMOG_STIFFNESS_OUTDIR
from wood_stiffness_homogenization.cli.run import run


widget_eval, eval_event = utilities.click_cli_widget(
    run,
    override_defaults=WOOD_SITFFNESS_HOMOG_DEFAULTS,
    title='Multiscale Wood Stiffness',
    description_text=(
        "This code computes the multiscale stiffness of native, polymer-infiltrated, and delignified + infiltrated "
        "wood. It works from the cell-wall constituents up to clearwood. It is a procedural port of `02_stiffness/"
        "hom_TranspWood_model.m`, with a lumped cell wall: the middle lamella (ML) and S2 layer are not separated."
    ),
    visible_panel_names=['Options'],
    button_description='Evaluate',
    button_running_text='Evaluating...',
    output_file_param='output',
)

# display(widget_eval)

visualizer_accordion = utilities.output_visualizer_widget(
    title="Wood Stiffness Results",
    file_format="csv",
    event_observer=eval_event,
    output_dir=HOMOG_STIFFNESS_OUTDIR,
    data_display_func=output_visualizer,
)
main_box = wdg.VBox([widget_eval, visualizer_accordion])
display(main_box)

In [ ]:
import utilities
utilities.final_spacer()